1.Data Collection: 

In [1]:
# Used the pandas library for storing the dataset as a python dataset and imported the CSV file.
import pandas as pd

df = pd.read_csv("student_placement_prediction_dataset_2026.csv")
print(df.shape)
df.head()

(100000, 26)


,student_id,age,gender,cgpa,branch,college_tier,internships_count,projects_count,certifications_count,coding_skill_score,...,mock_interview_score,attendance_percentage,backlogs,extracurricular_score,leadership_score,volunteer_experience,sleep_hours,study_hours_per_day,placement_status,salary_package_lpa
0,1,24,Male,7.53,IT,Tier 2,4,6,1,99.238568,...,72.647009,77.463863,2,63.382726,52.938240,Yes,6.7,3.6,Not Placed,0.00
1,2,21,Male,7.92,CSE,Tier 2,1,3,6,80.966123,...,61.699110,88.887600,1,73.694605,60.198856,No,4.4,2.3,Not Placed,0.00
2,3,22,Female,8.60,EEE,Tier 1,0,1,1,49.177184,...,87.396911,74.153265,0,63.329294,43.708803,No,8.8,5.9,Placed,11.99
3,4,24,Male,6.68,CSE,Tier 1,0,2,2,79.359084,...,58.401069,87.635955,1,47.636099,56.549154,Yes,8.1,4.4,Not Placed,0.00
4,5,20,Female,8.43,IT,Tier 3,1,4,3,65.018573,...,74.489201,79.120749,1,0.000000,67.268893,No,8.7,3.4,Placed,12.16


2.Data Preprocessing: 

In [2]:
# Checking the dataset to find missing values and duplicate rows so that the final training dataset does not contain 
# incomplete and redundant data.

print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

# Removing the identifier column (student_id) as it contains no useful data to determining the target variable and 
# salary_package_lpa as it leaks the final target variables value to the training dataset.
df = df.drop(columns=["student_id", "salary_package_lpa"])

Total missing values: 0
Duplicate rows: 0


In [3]:
# Encoding the text data into int data types so that model can perform faster using numbers rather than whole texts.

df["gender_encoded"] = (df["gender"] == "Male").astype(int) # 1 = male, 0 = female
df["volunteer_experience_encoded"] = (df["volunteer_experience"] == "Yes").astype(int) # 1 = yes, 0 = no
df["college_tier_encoded"] = df["college_tier"].str[-1].astype(int) # takes last character and turn into int as 1,2,3 (ordinal encoding)
df["encoded_placement_status"] = (df["placement_status"] == "Placed").astype(int) # 1 = placed, 0 = not placed
df = pd.concat([df, pd.get_dummies(df["branch"], dtype=int)], axis=1) # CSE, Civil, ECE, ... as seperate columns 1 if present (one hot encoding)
df = df.drop(columns=["gender", "volunteer_experience", "college_tier", "branch", "placement_status"]) # deleting the older columns without encoding

print(df.shape)
df.head()

(100000, 29)


,age,cgpa,internships_count,projects_count,certifications_count,coding_skill_score,aptitude_score,communication_skill_score,logical_reasoning_score,hackathons_participated,...,gender_encoded,volunteer_experience_encoded,college_tier_encoded,encoded_placement_status,CSE,Civil,ECE,EEE,IT,Mechanical
0,24,7.53,4,6,1,99.238568,81.707722,57.707166,59.070073,4,...,1,1,2,0,0,0,0,0,1,0
1,21,7.92,1,3,6,80.966123,63.116715,59.197085,78.976419,0,...,1,0,2,0,1,0,0,0,0,0
2,22,8.60,0,1,1,49.177184,48.658753,92.104885,80.603331,0,...,0,0,1,1,0,0,0,1,0,0
3,24,6.68,0,2,2,79.359084,66.376653,83.411798,64.187246,1,...,1,1,1,0,1,0,0,0,0,0
4,20,8.43,1,4,3,65.018573,61.274985,88.956331,56.163678,1,...,0,0,3,1,0,0,0,0,1,0


3.Feature Engineering:

In [4]:
# used sklearn library to split the data before any training so that the test data doesnt leak to the training data

from sklearn.model_selection import train_test_split, cross_val_score

# X as all columns except the target variable, y is the target varialble 
X = df.drop(columns="encoded_placement_status")
y = df["encoded_placement_status"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# used sklearn library to starndardize the data so that the model doesnt treat bigger numbers with higher priority
from sklearn.preprocessing import StandardScaler

# selected all continuous columns to standardize
scale_cols = ["cgpa", "coding_skill_score", "aptitude_score", "communication_skill_score",
              "logical_reasoning_score", "mock_interview_score", "attendance_percentage",
              "extracurricular_score", "leadership_score", "linkedin_connections"]

scaler = StandardScaler().fit(X_train[scale_cols]) # the scaler uses the train data to learn the means and standard deviation
for data in (X_train, X_test):
    data[["standardized_" + c for c in scale_cols]] = scaler.transform(data[scale_cols]) # assigns all standardized values to train and test datasets

# removes the old columns mentoned above as scale_cols so only one copy of the stadardized data exists
X_train = X_train.drop(columns=scale_cols)
X_test = X_test.drop(columns=scale_cols)

print(X_train.shape, X_test.shape)
X_train.head()

(80000, 28) (20000, 28)


,age,internships_count,projects_count,certifications_count,hackathons_participated,github_repos,backlogs,sleep_hours,study_hours_per_day,gender_encoded,...,standardized_cgpa,standardized_coding_skill_score,standardized_aptitude_score,standardized_communication_skill_score,standardized_logical_reasoning_score,standardized_mock_interview_score,standardized_attendance_percentage,standardized_extracurricular_score,standardized_leadership_score,standardized_linkedin_connections
12685,21,2,4,2,1,6,0,5.3,2.7,1,...,0.970132,-0.134162,1.052074,0.249635,-0.168017,0.004270,1.621905,0.018208,1.034121,0.814771
72956,19,0,4,4,1,5,2,7.7,2.8,1,...,1.332465,0.396304,0.804193,0.069629,-0.375126,0.616741,0.486152,-1.021458,-0.134366,-1.556373
75845,23,0,3,3,1,2,2,9.8,0.7,1,...,-1.032766,0.055630,-0.409284,0.671618,-0.001013,-2.048764,1.621905,-1.824987,0.131912,0.254716
83333,22,0,4,3,1,3,0,7.9,4.7,0,...,0.265595,0.036316,-1.761297,-1.590711,-0.055169,0.943701,-0.312126,0.130636,2.040287,1.382100
76658,19,5,3,0,1,3,2,3.5,1.8,0,...,2.016872,-1.455508,0.688710,0.166988,-1.011062,-0.343187,-1.726671,0.627365,-0.599046,1.436651


In [5]:
# use sklearn library to select the features that has importnace (gives a better estimation of the target variable) that is above avg 
# selected from random forest while training
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# selects featires with above avg importance
selector = SelectFromModel(RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
selector.fit(X_train, y_train)

# choose the featires that passed the random forest and puts it to a lost and anly keeps those columns in train and test datasets
selected = X_train.columns[selector.get_support()].tolist()
print("Selected features:", selected)
X_train, X_test = X_train[selected], X_test[selected]

print(X_train.shape, X_test.shape)
X_train.head()

Selected features: ['internships_count', 'projects_count', 'backlogs', 'standardized_cgpa', 'standardized_coding_skill_score', 'standardized_aptitude_score', 'standardized_communication_skill_score', 'standardized_logical_reasoning_score', 'standardized_mock_interview_score', 'standardized_attendance_percentage', 'standardized_extracurricular_score', 'standardized_leadership_score', 'standardized_linkedin_connections']
(80000, 13) (20000, 13)


,internships_count,projects_count,backlogs,standardized_cgpa,standardized_coding_skill_score,standardized_aptitude_score,standardized_communication_skill_score,standardized_logical_reasoning_score,standardized_mock_interview_score,standardized_attendance_percentage,standardized_extracurricular_score,standardized_leadership_score,standardized_linkedin_connections
12685,2,4,0,0.970132,-0.134162,1.052074,0.249635,-0.168017,0.004270,1.621905,0.018208,1.034121,0.814771
72956,0,4,2,1.332465,0.396304,0.804193,0.069629,-0.375126,0.616741,0.486152,-1.021458,-0.134366,-1.556373
75845,0,3,2,-1.032766,0.055630,-0.409284,0.671618,-0.001013,-2.048764,1.621905,-1.824987,0.131912,0.254716
83333,0,4,0,0.265595,0.036316,-1.761297,-1.590711,-0.055169,0.943701,-0.312126,0.130636,2.040287,1.382100
76658,5,3,2,2.016872,-1.455508,0.688710,0.166988,-1.011062,-0.343187,-1.726671,0.627365,-0.599046,1.436651


4.Model Selection:

In [6]:
# uses sklearn library to use grid serach cross validation on two algorithms and compare the results to chose the best model
# for this scenario
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

models = {
    "Logistic Regression": GridSearchCV(LogisticRegression(max_iter=1000),{"C": [0.01, 0.1, 1, 10]}, cv=5, scoring="roc_auc"),
    "Gradient Boosting": GridSearchCV(HistGradientBoostingClassifier(random_state=42),{"max_depth": [2, 3], "learning_rate": [0.05, 0.1]}, cv=5, scoring="roc_auc"),
}

5.Model Training:

In [7]:
# trains the two models only on the training sets and print the results including the ROC-AUC which shows how well
# each model predicts placed students above unplaced students

for name, m in models.items():
    m.fit(X_train, y_train)
    print(f"{name} - best settings: {m.best_params_} - CV ROC-AUC = {m.best_score_:.4f}")

Logistic Regression - best settings: {'C': 10} - CV ROC-AUC = 0.5856
Gradient Boosting - best settings: {'learning_rate': 0.1, 'max_depth': 2} - CV ROC-AUC = 0.5807


6.Model Evaluation:

In [8]:
# uses sklearn library to measure different metrics of the two models on the test data to find the best model and shows the numbers
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# tests the models accuray on placed students and the probability that is gets it right
rows = []
for name, m in models.items():
    pred, prob = m.predict(X_test), m.predict_proba(X_test)[:, 1]
    rows.append([name, accuracy_score(y_test, pred), f1_score(y_test, pred), roc_auc_score(y_test, prob)])

# puts the gathered data into a pandas table
results = pd.DataFrame(rows, columns=["Model", "Accuracy", "F1", "ROC-AUC"]).set_index("Model").round(4)
display(results)

# choses the best model based on the cross validation score
best_name = max(models, key=lambda n: models[n].best_score_)
best = models[best_name].best_estimator_
print("Best model:", best_name)

,Accuracy,F1,ROC-AUC
Model,,,
Logistic Regression,0.5714,0.6628,0.5857
Gradient Boosting,0.5659,0.6732,0.5811


Best model: Logistic Regression


7.Model Deployment:

In [9]:
# used joblib to save the pyhton objects as a file so that the model uses the same scaling and fetures when new data is submitted 
# after deployment, this data is used by app.py
import joblib

joblib.dump({"model": best, "scaler": scaler, "scale_cols": scale_cols, "features": selected},"student_placement_model.pkl")

['student_placement_model.pkl']